###Install Libraries

In [1]:
# ============================================================
# LAB DATA QUALITY MONITOR
# TASK 1: MOCK DATA GENERATION
# Aragen Life Sciences Assignment
# ============================================================

!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 3.8 MB/s eta 0:00:00


###Import Libraries

In [2]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker()

random.seed(42)
np.random.seed(42)

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


###Configuration

In [3]:
NUM_ROWS = 500000
labs = [
    "Analytical Lab A",
    "Analytical Lab B",
    "QC Lab",
    "Microbiology Lab",
    "Stability Lab",
    "Research Lab",
    "Formulation Lab"
]

instruments = [
    f"INST-LC-{str(i).zfill(3)}"
    for i in range(1, 16)
]

operators = [
    fake.name()
    for _ in range(25)
]

analytes = [
    "Ibuprofen",
    "Paracetamol",
    "Aspirin",
    "Caffeine",
    "Amoxicillin",
    "Metformin",
    "Atorvastatin",
    "Diclofenac",
    "Cetirizine",
    "Omeprazole",
    "Azithromycin",
    "Doxycycline"
]

units = [
    "mg/mL",
    "ppm",
    "ug/mL",
    "g/L"
]

statuses = [
    "Pass",
    "Fail",
    "Pending"
]

print("Configuration Loaded")

Configuration Loaded


###Base dataset generation

In [4]:
start_date = datetime(2025, 7, 1)
end_date = datetime(2026, 6, 30)

date_range_days = (end_date - start_date).days

sample_ids = [
    f"LAB-{str(i).zfill(6)}"
    for i in range(1, NUM_ROWS + 1)
]

experiment_dates = [
    start_date + timedelta(days=random.randint(0, date_range_days))
    for _ in range(NUM_ROWS)
]

measured_values = np.random.normal(
    loc=100,
    scale=15,
    size=NUM_ROWS
)

df = pd.DataFrame({
    "sample_id": sample_ids,
    "experiment_date": experiment_dates,
    "instrument_id": np.random.choice(instruments, NUM_ROWS),
    "lab_name": np.random.choice(labs, NUM_ROWS),
    "analyte_name": np.random.choice(analytes, NUM_ROWS),
    "measured_value": measured_values.round(2),
    "unit": np.random.choice(units, NUM_ROWS),
    "operator_name": np.random.choice(operators, NUM_ROWS),
    "status": np.random.choice(
        statuses,
        NUM_ROWS,
        p=[0.75, 0.15, 0.10]
    )
})

print(df.shape)
df.head()

(500000, 9)


,sample_id,experiment_date,instrument_id,lab_name,analyte_name,measured_value,unit,operator_name,status
0,LAB-000001,2026-05-24,INST-LC-010,Formulation Lab,Ibuprofen,107.45,g/L,Jason Poole,Pass
1,LAB-000002,2025-08-27,INST-LC-001,Analytical Lab A,Doxycycline,97.93,ug/mL,Chad Williams,Pass
2,LAB-000003,2025-07-13,INST-LC-005,Stability Lab,Diclofenac,109.72,ppm,Jason Poole,Pass
3,LAB-000004,2025-11-18,INST-LC-003,Analytical Lab A,Diclofenac,122.85,ppm,Jennifer Olson,Fail
4,LAB-000005,2025-11-03,INST-LC-008,Analytical Lab A,Doxycycline,96.49,ppm,Jennifer Jensen,Pending


###Generate recorder timestamp

In [5]:
recorded_dates = []

for exp_date in df["experiment_date"]:

    hours_added = random.randint(1, 12)

    recorded_dates.append(
        exp_date + timedelta(hours=hours_added)
    )

df["recorded_at"] = recorded_dates

print("Recorded timestamps generated")

Recorded timestamps generated


###Introdcution of 5% null values

In [6]:
null_columns = [
    "instrument_id",
    "operator_name",
    "unit",
    "measured_value"
]

for col in null_columns:

    idx = np.random.choice(
        df.index,
        int(0.05 * NUM_ROWS),
        replace=False
    )

    df.loc[idx, col] = np.nan

print("Null values inserted")

Null values inserted


###Introduction of 3% duplicate sample IDs

In [7]:
duplicate_rows = int(0.03 * NUM_ROWS)

duplicate_indices = np.random.choice(
    df.index,
    duplicate_rows,
    replace=False
)

duplicate_values = np.random.choice(
    df["sample_id"],
    duplicate_rows
)

df.loc[
    duplicate_indices,
    "sample_id"
] = duplicate_values

print("Duplicate IDs inserted")

Duplicate IDs inserted


###Outlier Introduction

In [8]:
outlier_rows = int(0.02 * NUM_ROWS)

outlier_indices = np.random.choice(
    df.index,
    outlier_rows,
    replace=False
)

half = outlier_rows // 2

df.loc[
    outlier_indices[:half],
    "measured_value"
] = -50

df.loc[
    outlier_indices[half:],
    "measured_value"
] = 99999

print("Outliers inserted")

Outliers inserted


###Format Inconsistencies

In [9]:
format_rows = int(0.03 * NUM_ROWS)

indices = np.random.choice(
    df.index,
    format_rows,
    replace=False
)

for idx in indices:

    val = df.loc[idx, "lab_name"]

    choice = random.choice([
        "upper",
        "lower"
    ])

    if choice == "upper":
        df.loc[idx, "lab_name"] = val.upper()

    else:
        df.loc[idx, "lab_name"] = val.lower()

print("Formatting issues inserted")

Formatting issues inserted


###Stale Timestamps

In [10]:
stale_rows = int(0.03 * NUM_ROWS)

stale_indices = np.random.choice(
    df.index,
    stale_rows,
    replace=False
)

for idx in stale_indices:

    df.loc[idx, "recorded_at"] = (
        df.loc[idx, "experiment_date"]
        + timedelta(days=random.randint(5, 20))
    )

print("Stale timestamps inserted")

Stale timestamps inserted


###Dataset Verification

In [11]:
print("\nDataset Shape")
print(df.shape)

print("\nNull Values")
print(df.isnull().sum())

print("\nDuplicate Sample IDs")
print(df["sample_id"].duplicated().sum())

print("\nSample Data")
df.sample(10)


Dataset Shape
(500000, 10)

Null Values
sample_id              0
experiment_date        0
instrument_id      25000
lab_name               0
analyte_name           0
measured_value     24507
unit               25000
operator_name      25000
status                 0
recorded_at            0
dtype: int64

Duplicate Sample IDs
14567

Sample Data


,sample_id,experiment_date,instrument_id,lab_name,analyte_name,measured_value,unit,operator_name,status,recorded_at
329467,LAB-329468,2025-12-31,INST-LC-009,Analytical Lab B,Omeprazole,85.69,g/L,Gina Green,Fail,2025-12-31 01:00:00
258153,LAB-258154,2026-06-19,INST-LC-006,Analytical Lab A,Metformin,96.26,mg/mL,Nancy Schwartz,Pass,2026-06-19 07:00:00
207705,LAB-207706,2025-09-06,INST-LC-001,Stability Lab,Cetirizine,112.84,mg/mL,Charles Cummings,Pass,2025-09-06 03:00:00
242024,LAB-242025,2025-07-02,INST-LC-009,Research Lab,Ibuprofen,103.35,mg/mL,Daniel Johnson,Pass,2025-07-02 12:00:00
228509,LAB-228510,2026-05-25,INST-LC-008,Microbiology Lab,Azithromycin,88.93,ppm,Denise Nolan,Pass,2026-05-25 12:00:00
268818,LAB-268819,2025-10-24,INST-LC-004,Analytical Lab B,Omeprazole,91.16,ppm,Tracy Simmons,Pass,2025-10-24 04:00:00
31794,LAB-031795,2025-09-07,INST-LC-006,Research Lab,Aspirin,71.71,ppm,Tracy Simmons,Pass,2025-09-19 00:00:00
128988,LAB-294254,2026-06-24,INST-LC-011,Formulation Lab,Aspirin,96.86,ppm,Christopher Thomas,Pending,2026-06-24 02:00:00
470999,LAB-471000,2025-12-01,INST-LC-007,Research Lab,Azithromycin,116.23,ug/mL,Tracy Simmons,Pass,2025-12-01 12:00:00
419936,LAB-419937,2025-12-14,INST-LC-002,Formulation Lab,Azithromycin,73.19,g/L,Daniel Johnson,Fail,2025-12-14 03:00:00


###Save Dataset

In [12]:
df.to_csv(
    "lab_data.csv",
    index=False
)

print("Dataset Saved Successfully")
print("File Name: lab_data.csv")

Dataset Saved Successfully
File Name: lab_data.csv


###Download CSV

In [13]:
from google.colab import files
files.download("lab_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
df.shape

(500000, 10)